In [0]:
# This is a python script which reads the Citibike DBFS feed every 5 minutes and writes the station status to json
# in the citibike_project volume S3 bucket.


import json, os, requests, gzip
from datetime import datetime, timezone


# Your professor is the one getting notifications if there is a problem

UA = {"User-Agent": "CUNY-SPS-DataScience-Archiver/1.0 (george.hagstrom@cuny.edu)"}
ROOT = "https://gbfs.citibikenyc.com/gbfs/gbfs.json" # This is the GBFS feed url
BASE = "/Volumes/citibike_project/citibike/raw/station_status" # This is where we are putting the jsons


# resolve_feed returns the url for the target feed. Most of the time we will use it to verify the
# station_status feed, but it can be used for other feeds in GBFS too for example the 

def resolve_feed(name="station_status"):
    root = requests.get(ROOT, headers=UA, timeout=30).json()
    data = root["data"]

    # This next line is a bit paranoid. I am told v3 of GBFS does not have the 'en' field, but V1 and V2 does use it.
    # Probably if there is a big update more things will break than just this, but who knows maybe not. Currently citibike is using
    # v1.1
    feeds = data["en"]["feeds"] if "en" in data else data["feeds"]  
    # Return the feed URLs
    return {f["name"]: f["url"] for f in feeds}[name]

# This gets the station_status feed and writes it to a json file in the citibike_project. The
# filenames will be structured temporally with year first, then month, then day, in UTC time. Important
# to note that while this makes the time variable "seamless" it means we will have to transform it to handle a variety
# of predictions.

# Here is an example of what a file would look like: 
# /Volumes/citibike_project/citibike/raw/station_status/year=2026/month=08/day=21/status_20260821T214037Z.json


def poll():
    resp = requests.get(resolve_feed(), headers=UA, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    now = datetime.now(timezone.utc)
    d = f"{BASE}/year={now:%Y}/month={now:%m}/day={now:%d}"
    os.makedirs(d, exist_ok=True)
    
    path = f"{d}/status_{now:%Y%m%dT%H%M%S}Z.json.gz"
    with gzip.open(path, "wt") as f:
        json.dump({"fetched_at": now.isoformat(),
                "feed_last_updated": payload.get("last_updated"),
                "payload": payload}, f)
        
    
    return path

print(poll())